In [1]:
import pandas as pd

In [2]:
df = pd.read_pickle("swissprot_2022_04_with_darkmatter.pkl")

In [3]:
df.loc[df["dark_matter"]]

In [4]:
dark_names = df.loc[df["dark_matter"], "proteins"]

In [5]:
swdf = pd.read_pickle("swissprot.pkl")

In [6]:
swdf.loc[swdf["proteins"].isin(dark_names)]

In [7]:
sw25df = pd.read_pickle("swissprot_2025_03.pkl")

In [8]:
sw25_has_anno = sw25df.loc[sw25df["exp_annotations"].apply(lambda xs: len(xs) > 0)]

In [9]:
sw25_has_anno.shape

In [10]:
dark_sw25_has_anno = sw25_has_anno.loc[sw25_has_anno["proteins"].isin(dark_names)]

In [11]:
dark_sw25_has_anno

In [12]:
dark_sw25_has_anno.loc[dark_sw25_has_anno["proteins"].isin(swdf["proteins"])]

In [13]:
sw24df = pd.read_pickle("swissprot_2022_04.pkl")

In [14]:
weak_sw24df = sw24df.loc[!sw24df["proteins"].isin(swdf["proteins"])]

In [15]:
weak_sw24df = sw24df.loc[~sw24df["proteins"].isin(swdf["proteins"])]

In [16]:
weak_sw24df

In [17]:
dark_weak_sw24df = weak_sw24df.loc[weak_sw24df["proteins"].isin(dark_names)]

In [18]:
dark_weak_sw24df

In [19]:
dark_weak_sw24df.columns

In [20]:
dark_weak_sw24df["annotations"]

In [21]:
sw25_has_anno

In [22]:
dark_weak_24names = dark_weak_sw24df["proteins"]

In [23]:
sw25_has_anno_names = sw25_has_anno["proteins"]

In [24]:
dark_weak_24names.loc[dark_weak_24names.isin(sw25_has_anno_names)]

In [25]:

import pandas as pd

# 2022 dark-weak proteins
d22 = dark_weak_sw24df.copy()

# 2025 UniProtKB annotations
d25 = sw25df.copy()  # 替换为实际的 2025 数据表

def primary_acc(x):
    if pd.isna(x):
        return ""
    return str(x).split(";")[0].strip()

def go_set(x):
    if isinstance(x, (list, tuple, set)):
        return set(x)
    return set()

for d in (d22, d25):
    d["primary_acc"] = d["accessions"].map(primary_acc)

assert d22["primary_acc"].is_unique
assert d25["primary_acc"].is_unique

merged = d22.merge(
    d25,
    on="primary_acc",
    how="left",
    suffixes=("_22", "_25"),
    indicator=True,
    validate="one_to_one",
)

matched = merged.loc[
    merged["_merge"].eq("both")
].copy()

matched["exp22"] = matched[
    "exp_annotations_22"
].map(go_set)

matched["exp25"] = matched[
    "exp_annotations_25"
].map(go_set)

matched["new_exp_go"] = matched.apply(
    lambda r: sorted(r["exp25"] - r["exp22"]),
    axis=1
)

matched["has_new_exp"] = matched[
    "new_exp_go"
].map(bool)

print("2022 dark-weak:", len(d22))
print("Matched in 2025:", len(matched))
print("Unmatched:", len(d22) - len(matched))
print(
    "New experimental GO:",
    matched["has_new_exp"].sum()
)
print(
    "Name changed:",
    (
        matched["proteins_22"]
        != matched["proteins_25"]
    ).sum()
)

In [26]:
print("2025 total rows:", len(d25))
print("Unique primary accessions:", d25["primary_acc"].nunique())
print("Duplicated rows:", d25["primary_acc"].duplicated().sum())
print("Empty accessions:", d25["primary_acc"].eq("").sum())

# 重复最多的 accession
print(
    d25["primary_acc"]
    .value_counts()
    .head(20)
)

# 展示重复记录
dup25 = d25.loc[
    d25["primary_acc"].duplicated(keep=False)
].sort_values("primary_acc")

In [27]:
display(
    dup25[
        ["proteins", "primary_acc",
         "accessions", "exp_annotations"]
    ].head(30)
)

In [28]:
import pandas as pd

d22 = dark_weak_sw24df[
    ["proteins", "accessions", "cafa_target"]
].copy().reset_index(drop=True).reset_index(names="row22")

d25 = sw25df[
    ["proteins", "accessions"]
].copy().reset_index(drop=True).reset_index(names="row25")

anno25_names = set(sw25_has_anno_names.dropna())
d25["has_anno25"] = d25["proteins"].isin(anno25_names)


def expand_accessions(frame, row_id):
    x = frame[[row_id, "accessions"]].copy()
    x["accession"] = x["accessions"].fillna("").str.split(";")
    x = x.explode("accession")
    x["accession"] = x["accession"].str.strip()

    return x.loc[x["accession"].ne("")].drop_duplicates(
        [row_id, "accession"]
    )[[row_id, "accession"]]


a22 = expand_accessions(d22, "row22")
a25 = expand_accessions(d25, "row25")

# 所有 accession，包括 secondary accession
pairs = a22.merge(a25, on="accession", how="inner")

# 消除同一条目因多个 accession 产生的重复匹配
pairs = pairs[["row22", "row25"]].drop_duplicates()

audit = pairs.merge(
    d25[["row25", "proteins", "has_anno25"]],
    on="row25",
    how="left"
)

summary = audit.groupby("row22").agg(
    n_2025_records=("row25", "nunique"),
    has_anno25=("has_anno25", "any")
)

old_names = set(d22["proteins"])
new_annotated_names = set(
    d25.loc[d25["has_anno25"], "proteins"]
)

name_hits = old_names & new_annotated_names

print("2022 dark weak:", len(d22))
print("Name-matched annotated:", len(name_hits))
print("Accession-matched records:", len(summary))
print("Accession-matched annotated:",
      int(summary["has_anno25"].sum()))
print("Ambiguous matches:",
      int((summary["n_2025_records"] > 1).sum()))

In [29]:
import pandas as pd
import numpy as np

d22 = dark_weak_sw24df.copy().reset_index(drop=True)
d25 = sw25df.copy().reset_index(drop=True)

d22["rid22"] = d22.index
d25["rid25"] = d25.index

anno25_names = set(pd.Series(sw25_has_anno_names).dropna())
d25["has_anno25"] = d25["proteins"].isin(anno25_names)

for frame in (d22, d25):
    frame["primary"] = (
        frame["accessions"]
        .fillna("")
        .str.split(";")
        .str[0]
        .str.strip()
    )


def explode_accessions(frame, rid, flag):
    x = frame[[rid, "primary", "accessions"]].copy()
    x["accession"] = (
        x["accessions"].fillna("").str.split(";")
    )
    x = x.explode("accession")
    x["accession"] = x["accession"].str.strip()

    x = x[x["accession"].ne("")]
    x = x.drop_duplicates([rid, "accession"])
    x[flag] = x["accession"].eq(x["primary"])

    return x[[rid, "accession", flag]]


a22 = explode_accessions(d22, "rid22", "primary22")
a25 = explode_accessions(d25, "rid25", "primary25")

pairs = a22.merge(a25, on="accession", how="inner")

# 优先级：
# 4: primary22 -> primary25
# 3: primary22 -> secondary25
# 2: secondary22 -> primary25
# 1: secondary22 -> secondary25

pairs["priority"] = np.select(
    [
        pairs["primary22"] & pairs["primary25"],
        pairs["primary22"],
        pairs["primary25"],
    ],
    [4, 3, 2],
    default=1
)

# 同一对蛋白可能通过多个 accession 匹配
pairs = (
    pairs.sort_values("priority", ascending=False)
    .drop_duplicates(["rid22", "rid25"])
)

pairs = pairs.merge(
    d25[["rid25", "proteins", "has_anno25"]].rename(
        columns={"proteins": "name25"}
    ),
    on="rid25",
    how="left"
)

# 原始 any-accession 统计
raw_count = pairs.groupby("rid22")["rid25"].nunique()
raw_annotated = pairs.groupby("rid22")["has_anno25"].any()

# 只保留每个2022蛋白的最高优先级匹配
best = pairs[
    pairs["priority"].eq(
        pairs.groupby("rid22")["priority"].transform("max")
    )
].copy()

best_summary = best.groupby("rid22").agg(
    n_best=("rid25", "nunique"),
    has_anno25=("has_anno25", "any"),
    match_priority=("priority", "max"),
)

unique_best = best_summary["n_best"].eq(1)
annotated_best = best_summary["has_anno25"]

# 178与689的交集
name_ids = set(
    d22.loc[
        d22["proteins"].isin(anno25_names), "rid22"
    ]
)
accession_ids = set(
    raw_annotated[raw_annotated].index
)

print("=== Identity audit ===")
print("2022 dark weak:", len(d22))
print("2025 duplicate primary:",
      d25["primary"].duplicated().sum())
print("Primary-primary matches:",
      (best_summary["match_priority"] == 4).sum())
print("Raw ambiguous:", (raw_count > 1).sum())
print("Best-level ambiguous:", (~unique_best).sum())

print("\n=== Annotation audit ===")
print("Name annotated:", len(name_ids))
print("Any-accession annotated:", len(accession_ids))
print("Name & accession overlap:",
      len(name_ids & accession_ids))
print("Name-only annotated:",
      len(name_ids - accession_ids))
print("Accession-only annotated:",
      len(accession_ids - name_ids))

print("\n=== Priority-resolved annotation ===")
print("Unique-best annotated:",
      (unique_best & annotated_best).sum())
print("Primary-primary annotated:",
      (
          unique_best
          & annotated_best
          & (best_summary["match_priority"] == 4)
      ).sum())
print("Still ambiguous and annotated:",
      ((~unique_best) & annotated_best).sum())

In [30]:
# 查看重复 pri[200~# 查看重复 primary 的具体情况
dup25 = d25.loc[
    d25["primary"].duplicated(keep=False)
].copy()

print("Duplicated rows:", len(dup25))
print("Duplicated primary IDs:", dup25["primary"].nunique())
print("Empty primary:", d25["primary"].eq("").sum())

print(
    dup25.groupby("primary")
    .agg(
        rows=("rid25", "size"),
        names=("proteins", "nunique")
    )
    .sort_values("rows", ascending=False)
    .head(20)
)

In [31]:
strict_ids = set(
    best.loc[best["has_anno25"], "rid22"]
)

print("Name & strict:", len(name_ids & strict_ids))
print("Strict-only:", len(strict_ids - name_ids))
print("Name-only:", len(name_ids - strict_ids))

print("Any-accession excluded:",
      len(accession_ids - strict_ids))

In [32]:
import re
import pandas as pd

# 仅保留唯一最高优先级匹配且2025已有注释的记录
selected = best.loc[
    (best["priority"] == 4)
    & (best["has_anno25"])
].copy()

assert len(selected) == 207
assert selected["rid22"].is_unique
assert selected["rid25"].is_unique

cols = [
    "proteins", "primary", "accessions",
    "exp_annotations", "prop_annotations",
]

cols22 = [c for c in cols if c in d22.columns]
cols25 = [c for c in cols if c in d25.columns]

old = d22[
    ["rid22"] + cols22
].rename(
    columns={c: f"{c}_22" for c in cols22}
)

new = d25[
    ["rid25"] + cols25
].rename(
    columns={c: f"{c}_25" for c in cols25}
)

candidates = (
    selected[["rid22", "rid25", "priority"]]
    .merge(old, on="rid22", validate="one_to_one")
    .merge(new, on="rid25", validate="one_to_one")
)

GO_PATTERN = re.compile(r"GO:\d{7}")

def parse_go(value):
    if value is None:
        return set()
    return set(GO_PATTERN.findall(str(value)))

required = [
    "exp_annotations_22",
    "exp_annotations_25",
    "prop_annotations_22",
    "prop_annotations_25",
]

missing = [c for c in required if c not in candidates.columns]
if missing:
    raise ValueError(f"Missing GO columns: {missing}")

for suffix in ("22", "25"):
    for kind in ("exp", "prop"):
        col = f"{kind}_annotations_{suffix}"
        candidates[f"{kind}_go_{suffix}"] = (
            candidates[col].map(parse_go)
        )

candidates["new_exp_go"] = candidates.apply(
    lambda r: r["exp_go_25"] - r["exp_go_22"],
    axis=1
)

candidates["new_prop_go"] = candidates.apply(
    lambda r: r["prop_go_25"] - r["prop_go_22"],
    axis=1
)

candidates["n_new_exp_go"] = (
    candidates["new_exp_go"].map(len)
)
candidates["n_new_prop_go"] = (
    candidates["new_prop_go"].map(len)
)

print("Candidates:", len(candidates))
print("New experimental GO:",
      (candidates["n_new_exp_go"] > 0).sum())
print("New propagated GO:",
      (candidates["n_new_prop_go"] > 0).sum())

print("Unique new experimental terms:",
      len(set().union(*candidates["new_exp_go"])))

candidates.to_pickle(
    "dark_weak_2022_2025_go_candidates.pkl"
)

In [33]:
for acc in ["Q2MC65", "Q55DZ5", "Q9S7X3"]:
    print(f"\n=== {acc} ===")
    print(
        d25.loc[
            d25["primary"].eq(acc),
            ["rid25", "proteins", "accessions"]
        ].to_string(index=False)
    )

duplicate_primaries = set(
    d25.loc[
        d25["primary"].duplicated(keep=False),
        "primary"
    ]
)

candidate_primaries = set(candidates["primary_25"])

print(
    "Duplicate primary IDs in 207 candidates:",
    len(duplicate_primaries & candidate_primaries)
)

In [34]:
from goatools.obo_parser import GODag
from collections import Counter

# 修改成项目中2022年使用的GO [200~from goatools.obo_parser import GODag
from collections import Counter

# 修改成项目中2022年使用的GO [200~from goatools.obo_parser import GODag
from collections import Counter

# 修改成项目中2022年使用的GO [200~from goatools.obo_parser import GODag
from collections import Counter

# 修改成项目中2022年使用的GO ontology路径
go = GODag("go.obo")

namespaces = {
    "biological_process": "BP",
    "molecular_function": "MF",
    "cellular_component": "CC",
}

rows = []

for _, r in candidates.iterrows():
    for go_id in r["exp_go_25"]:
        namespace = (
            namespaces.get(go[go_id].namespace, "Other")
            if go_id in go else "Unmapped"
        )

        rows.append({
            "protein": r["proteins_22"],
            "accession": r["primary_22"],
            "go_id": go_id,
            "namespace": namespace
        })

anno = pd.DataFrame(rows).drop_duplicates()

summary = (
    anno.groupby("namespace")
    .agg(
        proteins=("protein", "nunique"),
        unique_go=("go_id", "nunique"),
        positive_pairs=("go_id", "size")
    )
)

print(summary)
print("Total unique GO:", anno["go_id"].nunique())
print("Total positive pairs:", len(anno))
print("Unmapped GO IDs:",
      anno.loc[anno["namespace"] == "Unmapped", "go_id"]
          .drop_duplicates().tolist())

In [35]:
print("2022 sequence columns:",
      [c for c in d22.columns if "seq" in c.lower()])

print("2025 sequence columns:",
      [c for c in d25.columns if "seq" in c.lower()])

In [36]:
import re
import pandas as pd
import hashlib

seq22 = d22[["rid22", "sequences"]].rename(
    columns={"sequences": "sequence_22"}
)

seq25 = d25[["rid25", "sequences"]].rename(
    columns={"sequences": "sequence_25"}
)

audit = (
    candidates
    .merge(seq22, on="rid22", validate="one_to_one")
    .merge(seq25, on="rid25", validate="one_to_one")
)

# 不把任意对象的字符串表示误认为蛋白序列
AA_PATTERN = re.compile(r"[ACDEFGHIKLMNPQRSTVWYBXZJUO]+")

def normalize_sequence(value):
    if not isinstance(value, str):
        return None

    seq = re.sub(r"\s+", "", value).upper().rstrip("*")

    if not seq or not AA_PATTERN.fullmatch(seq):
        return None

    return seq

for year in ("22", "25"):
    audit[f"seq_{year}"] = audit[
        f"sequence_{year}"
    ].map(normalize_sequence)

    audit[f"length_{year}"] = (
        audit[f"seq_{year}"].map(
            lambda s: len(s) if s is not None else None
        )
    )

    audit[f"sha256_{year}"] = (
        audit[f"seq_{year}"].map(
            lambda s: hashlib.sha256(s.encode()).hexdigest()
            if s is not None else None
        )
    )

def sequence_status(row):
    if row["seq_22"] is None or row["seq_25"] is None:
        return "unresolved"

    if row["seq_22"] == row["seq_25"]:
        return "identical"

    return "changed"

audit["sequence_status"] = audit.apply(
    sequence_status, axis=1
)

audit["name_changed"] = (
    audit["proteins_22"] != audit["proteins_25"]
)

print("=== Sequence audit ===")
print("Total candidates:", len(audit))
print(audit["sequence_status"].value_counts())
print("Entry name changed:", audit["name_changed"].sum())

print("\n=== Original 178 vs additional 29 ===")
audit["original_name_match"] = audit["rid22"].isin(name_ids)

print(
    pd.crosstab(
        audit["original_name_match"],
        audit["sequence_status"],
        margins=True
    )
)

print("\n=== Name and sequence consistency ===")
print(
    pd.crosstab(
        audit["name_changed"],
        audit["sequence_status"],
        margins=True
    )
)

print("\n=== Changed sequence examples ===")
print(
    audit.loc[
        audit["sequence_status"] == "changed",
        [
            "proteins_22", "proteins_25",
            "primary_22", "primary_25",
            "length_22", "length_25"
        ]
    ].head(20).to_string(index=False)
)

audit.to_pickle(
    "dark_weak_207_sequence_audit.pkl"
)

In [37]:

import pandas as pd
import hashlib
import re

# ============================================================
# 1. 使用当前已完成的207个候选匹配
# ============================================================

selected = candidates[
    ["rid22", "rid25"]
].drop_duplicates().copy()

assert len(selected) == 207
assert selected["rid22"].is_unique
assert selected["rid25"].is_unique

cols = ["proteins", "accessions", "sequences"]

old = d22[
    ["rid22"] + cols
].rename(columns={
    "proteins": "protein_22",
    "accessions": "accessions_22",
    "sequences": "sequence_22"
})

new = d25[
    ["rid25"] + cols
].rename(columns={
    "proteins": "protein_25",
    "accessions": "accessions_25",
    "sequences": "sequence_25"
})

identity = (
    selected
    .merge(old, on="rid22", validate="one_to_one")
    .merge(new, on="rid25", validate="one_to_one")
)

# ============================================================
# 2. 标准化氨基酸序列
# ============================================================

AA_PATTERN = re.compile(r"[ACDEFGHIKLMNPQRSTVWYBXZJUO]+")

def normalize_seq(value):
    if not isinstance(value, str):
        return None

    seq = "".join(value.split()).upper().rstrip("*")

    if not seq or not AA_PATTERN.fullmatch(seq):
        return None

    return seq


identity["seq22"] = identity["sequence_22"].map(normalize_seq)
identity["seq25"] = identity["sequence_25"].map(normalize_seq)

def sha256_seq(seq):
    if seq is None:
        return None
    return hashlib.sha256(seq.encode("ascii")).hexdigest()

identity["sha256_22"] = identity["seq22"].map(sha256_seq)
identity["sha256_25"] = identity["seq25"].map(sha256_seq)

identity["length_22"] = identity["seq22"].map(
    lambda s: len(s) if s is not None else None
)

identity["length_25"] = identity["seq25"].map(
    lambda s: len(s) if s is not None else None
)

# ============================================================
# 3. 序列和名称一致性检查
# ============================================================

valid = (
    identity["seq22"].notna()
    & identity["seq25"].notna()
)

identity["sequence_identical"] = (
    valid
    & identity["seq22"].eq(identity["seq25"])
)

identity["sequence_status"] = "unresolved"
identity.loc[valid, "sequence_status"] = "changed"
identity.loc[
    identity["sequence_identical"],
    "sequence_status"
] = "identical"

identity["name_changed"] = (
    identity["protein_22"] != identity["protein_25"]
)

identity["accession_field_changed"] = (
    identity["accessions_22"] != identity["accessions_25"]
)

# 注意：accession_field_changed 只比较原始字段，
# 不能据此判断真实 primary accession 是否改变。

# ============================================================
# 4. 输出审计结果
# ============================================================

print("=== Historical identity audit ===")
print("Total:", len(identity))
print(identity["sequence_status"].value_counts())

print("\n=== Entry name changed ===")
print(identity["name_changed"].value_counts())

print("\n=== Cross table ===")
print(
    pd.crosstab(
        identity["name_changed"],
        identity["sequence_status"],
        margins=True
    )
)

print("\n=== Sequence changed proteins ===")
print(
    identity.loc[
        identity["sequence_status"] == "changed",
        [
            "protein_22", "protein_25",
            "length_22", "length_25",
            "accessions_22", "accessions_25"
        ]
    ].to_string(index=False)
)

# ============================================================
# 5. 保存结果
# ============================================================

strict = identity.loc[
    identity["sequence_status"] == "identical"
].copy()

changed = identity.loc[
    identity["sequence_status"] == "changed"
].copy()

identity.to_pickle("dark_weak_identity_207.pkl")
strict.to_pickle("dark_weak_seq_identical.pkl")
changed.to_pickle("dark_weak_seq_changed.pkl")

summary_cols = [
    "rid22", "rid25",
    "protein_22", "protein_25",
    "accessions_22", "accessions_25",
    "length_22", "length_25",
    "sha256_22", "sha256_25",
    "sequence_status",
    "name_changed",
    "accession_field_changed"
]

identity[summary_cols].to_csv(
    "dark_weak_identity_audit.csv",
    index=False
)

print("\nSequence-invariant candidates:", len(strict))
print("Sequence-changed candidates:", len(changed))

In [38]:
# 检查严格候选在2025数据库中的完全相同序列
target_sequences = set(strict["seq25"])

same_seq_25 = d25.loc[
    d25["sequences"].isin(target_sequences),
    ["proteins", "sequences"]
].copy()

seq_counts = (
    same_seq_25.groupby("sequences")["proteins"]
    .nunique()
)

strict["n_2025_same_sequence"] = (
    strict["seq25"].map(seq_counts).fillna(0).astype(int)
)

print("=== Sequence uniqueness ===")
print(strict["n_2025_same_sequence"].value_counts().sort_index())

print("\nCandidates with nonunique sequences:")
print(
    strict.loc[
        strict["n_2025_same_sequence"] > 1,
        [
            "protein_22",
            "protein_25",
            "length_22",
            "n_2025_same_sequence"
        ]
    ].to_string(index=False)
)

In [39]:

import pandas as pd
import hashlib
import re

# ============================================================
# 1. 使用当前已完成的207个候选匹配
# ============================================================

selected = candidates[
    ["rid22", "rid25"]
].drop_duplicates().copy()

assert len(selected) == 207
assert selected["rid22"].is_unique
assert selected["rid25"].is_unique

cols = ["proteins", "accessions", "sequences"]

old = d22[
    ["rid22"] + cols
].rename(columns={
    "proteins": "protein_22",
    "accessions": "accessions_22",
    "sequences": "sequence_22"
})

new = d25[
    ["rid25"] + cols
].rename(columns={
    "proteins": "protein_25",
    "accessions": "accessions_25",
    "sequences": "sequence_25"
})

identity = (
    selected
    .merge(old, on="rid22", validate="one_to_one")
    .merge(new, on="rid25", validate="one_to_one")
)

# ============================================================
# 2. 标准化氨基酸序列
# ============================================================

AA_PATTERN = re.compile(r"[ACDEFGHIKLMNPQRSTVWYBXZJUO]+")

def normalize_seq(value):
    if not isinstance(value, str):
        return None

    seq = "".join(value.split()).upper().rstrip("*")

    if not seq or not AA_PATTERN.fullmatch(seq):
        return None

    return seq


identity["seq22"] = identity["sequence_22"].map(normalize_seq)
identity["seq25"] = identity["sequence_25"].map(normalize_seq)

def sha256_seq(seq):
    if seq is None:
        return None
    return hashlib.sha256(seq.encode("ascii")).hexdigest()

identity["sha256_22"] = identity["seq22"].map(sha256_seq)
identity["sha256_25"] = identity["seq25"].map(sha256_seq)

identity["length_22"] = identity["seq22"].map(
    lambda s: len(s) if s is not None else None
)

identity["length_25"] = identity["seq25"].map(
    lambda s: len(s) if s is not None else None
)

# ============================================================
# 3. 序列和名称一致性检查
# ============================================================

valid = (
    identity["seq22"].notna()
    & identity["seq25"].notna()
)

identity["sequence_identical"] = (
    valid
    & identity["seq22"].eq(identity["seq25"])
)

identity["sequence_status"] = "unresolved"
identity.loc[valid, "sequence_status"] = "changed"
identity.loc[
    identity["sequence_identical"],
    "sequence_status"
] = "identical"

identity["name_changed"] = (
    identity["protein_22"] != identity["protein_25"]
)

identity["accession_field_changed"] = (
    identity["accessions_22"] != identity["accessions_25"]
)

# 注意：accession_field_changed 只比较原始字段，
# 不能据此判断真实 primary accession 是否改变。

# ============================================================
# 4. 输出审计结果
# ============================================================

print("=== Historical identity audit ===")
print("Total:", len(identity))
print(identity["sequence_status"].value_counts())

print("\n=== Entry name changed ===")
print(identity["name_changed"].value_counts())

print("\n=== Cross table ===")
print(
    pd.crosstab(
        identity["name_changed"],
        identity["sequence_status"],
        margins=True
    )
)

print("\n=== Sequence changed proteins ===")
print(
    identity.loc[
        identity["sequence_status"] == "changed",
        [
            "protein_22", "protein_25",
            "length_22", "length_25",
            "accessions_22", "accessions_25"
        ]
    ].to_string(index=False)
)

# ============================================================
# 5. 保存结果
# ============================================================

strict = identity.loc[
    identity["sequence_status"] == "identical"
].copy()

changed = identity.loc[
    identity["sequence_status"] == "changed"
].copy()

identity.to_pickle("dark_weak_identity_207.pkl")
strict.to_pickle("dark_weak_seq_identical.pkl")
changed.to_pickle("dark_weak_seq_changed.pkl")

summary_cols = [
    "rid22", "rid25",
    "protein_22", "protein_25",
    "accessions_22", "accessions_25",
    "length_22", "length_25",
    "sha256_22", "sha256_25",
    "sequence_status",
    "name_changed",
    "accession_field_changed"
]

identity[summary_cols].to_csv(
    "dark_weak_identity_audit.csv",
    index=False
)

print("\nSequence-invariant candidates:", len(strict))
print("Sequence-changed candidates:", len(changed))

In [40]:

import pandas as pd
import re

GO_RE = re.compile(r"GO:\d{7}")

def get_go_ids(value):
    if value is None:
        return set()
    return set(GO_RE.findall(str(value)))

# strict: 197 sequence-invariant candidates
# d25: complete 2025 Swiss-Prot DataFrame

target_sequences = set(strict["seq25"])

twins25 = d25.loc[
    d25["sequences"].isin(target_sequences)
].copy()

groups25 = {
    seq: group
    for seq, group in twins25.groupby("sequences")
}

records = []

for _, row in strict.iterrows():
    seq = row["seq25"]
    group = groups25[seq]

    # 该候选实际匹配到的2025条目
    target = group.loc[group["rid25"].eq(row["rid25"])]
    assert len(target) == 1

    target_go = get_go_ids(
        target.iloc[0]["exp_annotations"]
    )

    go_sets = [
        get_go_ids(v)
        for v in group["exp_annotations"]
    ]

    all_go = set().union(*go_sets)

    records.append({
        "protein_22": row["protein_22"],
        "protein_25": row["protein_25"],
        "rid22": row["rid22"],
        "rid25": row["rid25"],
        "seq25": seq,
        "n_2025_same_seq": group["proteins"].nunique(),
        "n_2025_taxa": (
            group["orgs"].nunique()
            if "orgs" in group.columns else None
        ),
        "n_go_profiles": len({
            frozenset(g) for g in go_sets
        }),
        "target_exp_go": sorted(target_go),
        "other_extra_exp_go": sorted(all_go - target_go),
        "same_seq_names": ";".join(
            group["proteins"].astype(str)
        ),
    })

duplicate_audit = pd.DataFrame(records)

print("=== Sequence cluster audit ===")
print("Candidates:", len(duplicate_audit))
print(
    "Distinct candidate sequences:",
    duplicate_audit["seq25"].nunique()
)

print(
    "Candidates with multiple 2025 entries:",
    (duplicate_audit["n_2025_same_seq"] > 1).sum()
)

print(
    "Candidates with differing GO profiles:",
    (
        (duplicate_audit["n_2025_same_seq"] > 1)
        & (duplicate_audit["n_go_profiles"] > 1)
    ).sum()
)

print("\n=== Nonunique sequences ===")

print(
    duplicate_audit.loc[
        duplicate_audit["n_2025_same_seq"] > 1,
        [
            "protein_25",
            "n_2025_same_seq",
            "n_2025_taxa",
            "n_go_profiles",
            "other_extra_exp_go",
            "same_seq_names"
        ]
    ].to_string(index=False)
)

duplicate_audit.to_pickle(
    "dark_weak_197_sequence_clusters.pkl"
)

In [41]:
# df: complete 2022_04 Swiss-Prot DataFrame
# strict: 197 historical sequence-invariant candidates
# d22, d25: previously constructed DataFrames

all22_same_seq = df.loc[
    df["sequences"].isin(set(strict["seq22"]))
].copy()

historical_exp_by_seq = {}
historical_all_by_seq = {}

for seq, group in all22_same_seq.groupby("sequences"):
    historical_exp_by_seq[seq] = set().union(
        *(get_go_ids(v) for v in group["exp_annotations"])
    )
    historical_all_by_seq[seq] = set().union(
        *(get_go_ids(v) for v in group["annotations"])
    )

old_lookup = d22.set_index("rid22")
new_lookup = d25.set_index("rid25")

rows = []

for _, r in strict.iterrows():
    seq = r["seq22"]
    rid22 = r["rid22"]
    rid25 = r["rid25"]

    go25 = get_go_ids(
        new_lookup.at[rid25, "exp_annotations"]
    )

    self_go22 = get_go_ids(
        old_lookup.at[rid22, "annotations"]
    )

    same_seq_exp22 = historical_exp_by_seq.get(
        seq, set()
    )

    same_seq_all22 = historical_all_by_seq.get(
        seq, set()
    )

    for go_id in sorted(go25):
        rows.append({
            "protein": r["protein_22"],
            "rid22": rid22,
            "go_id": go_id,
            "self_go_known_2022": go_id in self_go22,
            "identical_seq_exp_2022":
                go_id in same_seq_exp22,
            "identical_seq_any_2022":
                go_id in same_seq_all22,
        })

go_evidence_audit = pd.DataFrame(rows)

go_evidence_audit["candidate_novel_direct"] = (
    ~go_evidence_audit["identical_seq_any_2022"]
)

print("=== Historical GO evidence audit ===")
print("Protein-GO pairs:", len(go_evidence_audit))

for col in [
    "self_go_known_2022",
    "identical_seq_exp_2022",
    "identical_seq_any_2022",
    "candidate_novel_direct"
]:
    print(col, int(go_evidence_audit[col].sum()))

print(
    "\nProteins with at least one candidate-novel GO:",
    go_evidence_audit.groupby("rid22")[
        "candidate_novel_direct"
    ].any().sum()
)

go_evidence_audit.to_pickle(
    "dark_weak_197_historical_go_evidence.pkl"
)

In [42]:
dark_weak_sw24df.to_pickle(
    "data/dark_weak_2022_04.pkl"
)

In [43]:
dark_weak_sw24df.to_pickle(
    "./dark_weak_2022_04.pkl"
)

In [44]:

from pathlib import Path
import pandas as pd

root = Path("data/dark_weak_test")

pairs = pd.read_csv(
    root / "tables/future_go_pairs.tsv",
    sep="\t"
)

# 读取模型真实 GO 词表映射
vocab = {}

for ns in ("bp", "mf", "cc"):
    mapping = pd.read_csv(
        root / "labels" / ns / "vocabulary_mapping.tsv",
        sep="\t"
    )
    vocab[ns] = set(mapping["canonical_go_id_2022"])

# 是否可被现有模型直接输出
pairs["in_model_vocab"] = pairs.apply(
    lambda r: (
        r["namespace"] in vocab
        and r["canonical_go_2022"] in
            vocab.get(r["namespace"], set())
    ),
    axis=1
)

# 历史未记录 + 严格实验 + 可映射 + 模型可预测
mask = (
    pairs["strict_eligible"]
    & pairs["is_strict_experimental"]
    & pairs["historically_unrecorded_identical_seq"]
    & pairs["in_model_vocab"]
)

discovery = pairs.loc[mask].copy()

summary = (
    discovery.groupby("namespace")
    .agg(
        proteins=("rid22", "nunique"),
        unique_go=("canonical_go_2022", "nunique"),
        positive_pairs=("canonical_go_2022", "size")
    )
)

print("=== Experimental Historical-Novel ===")
print(summary)

print("\nTotal unique proteins:",
      discovery["rid22"].nunique())
print("Total positive pairs:", len(discovery))

discovery.to_csv(
    root / "tables/"
           "experimental_historical_novel_pairs.tsv",
    sep="\t",
    index=False
)

summary.to_csv(
    root / "tables/"
           "experimental_historical_novel_summary.tsv",
    sep="\t"
)

In [45]:

from pathlib import Path
import pandas as pd

root = Path("./dark_weak_test")

pairs = pd.read_csv(
    root / "tables/future_go_pairs.tsv",
    sep="\t"
)

# 读取模型真实 GO 词表映射
vocab = {}

for ns in ("bp", "mf", "cc"):
    mapping = pd.read_csv(
        root / "labels" / ns / "vocabulary_mapping.tsv",
        sep="\t"
    )
    vocab[ns] = set(mapping["canonical_go_id_2022"])

# 是否可被现有模型直接输出
pairs["in_model_vocab"] = pairs.apply(
    lambda r: (
        r["namespace"] in vocab
        and r["canonical_go_2022"] in
            vocab.get(r["namespace"], set())
    ),
    axis=1
)

# 历史未记录 + 严格实验 + 可映射 + 模型可预测
mask = (
    pairs["strict_eligible"]
    & pairs["is_strict_experimental"]
    & pairs["historically_unrecorded_identical_seq"]
    & pairs["in_model_vocab"]
)

discovery = pairs.loc[mask].copy()

summary = (
    discovery.groupby("namespace")
    .agg(
        proteins=("rid22", "nunique"),
        unique_go=("canonical_go_2022", "nunique"),
        positive_pairs=("canonical_go_2022", "size")
    )
)

print("=== Experimental Historical-Novel ===")
print(summary)

print("\nTotal unique proteins:",
      discovery["rid22"].nunique())
print("Total positive pairs:", len(discovery))

discovery.to_csv(
    root / "tables/"
           "experimental_historical_novel_pairs.tsv",
    sep="\t",
    index=False
)

summary.to_csv(
    root / "tables/"
           "experimental_historical_novel_summary.tsv",
    sep="\t"
)

In [46]:

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse

root = Path("./dark_weak_test")

expected = {
    "bp": (92, 154),
    "mf": (39, 57),
    "cc": (84, 114),
}

all_positive_rows = set()

for ns in ("bp", "mf", "cc"):
    folder = root / "labels" / ns

    exp = sparse.load_npz(
        folder / "strict_experimental_direct.npz"
    ).astype(bool)

    novel = sparse.load_npz(
        folder / "historically_unrecorded_direct.npz"
    ).astype(bool)

    assert exp.shape == novel.shape

    # 共同满足实验性证据与历史未记录条件
    y = exp.multiply(novel).astype(np.uint8).tocsr()
    y.eliminate_zeros()

    # 原始模型列可能包含相同canonical GO的多个别名
    mapping = pd.read_csv(
        folder / "vocabulary_mapping.tsv", sep="\t"
    )
    canonical = mapping["canonical_go_id_2022"].to_numpy()

    row_idx, col_idx = y.nonzero()

    unique_pairs = {
        (int(r), str(canonical[c]))
        for r, c in zip(row_idx, col_idx)
    }

    positive_rows = set(row_idx.tolist())
    all_positive_rows.update(positive_rows)

    print(f"\n=== {ns.upper()} ===")
    print("Matrix shape:", y.shape)
    print("Positive proteins:", len(positive_rows))
    print("Model-column pairs:", y.nnz)
    print("Canonical GO pairs:", len(unique_pairs))

    assert (len(positive_rows), len(unique_pairs)) == expected[ns]

    # 保持全部197行和原模型列顺序
    sparse.save_npz(
        folder / "experimental_historical_novel_direct.npz",
        y,
        compressed=True
    )

    # 各namespace实际可评价的蛋白行
    np.save(
        folder / "experimental_historical_novel_rows.npy",
        np.array(sorted(positive_rows), dtype=np.int64)
    )

print("\nUnion of evaluated proteins:", len(all_positive_rows))
assert len(all_positive_rows) == 161

In [47]:
from pathlib import Path
import pandas as pd

# 兼容在项目根目录或 data/ 目录启动 IPython
root = Path("data/dark_weak_test/interpro2go_probe")
if not root.exists():
    root = Path("dark_weak_test/interpro2go_probe")

proteins = pd.read_csv(
    root / "protein_summary.tsv",
    sep="\t"
)

pairs = pd.read_csv(
    root / "interpro2go_pairs.tsv",
    sep="\t"
)

print("=== InterPro coverage ===")
print("Matched proteins:", len(proteins))
print("2022 has IPR:", (proteins.n_ipr_22 > 0).sum())
print("2025 has IPR:", (proteins.n_ipr_25 > 0).sum())

print("2022 no IPR, 2025 has IPR:", (
    (proteins.n_ipr_22 == 0) &
    (proteins.n_ipr_25 > 0)
).sum())

print("Proteins with new IPR:", (
    proteins.n_new_ipr > 0
).sum())

print("Proteins with added GO:", (
    proteins.n_added_direct_go > 0
).sum())

print("\n=== GO source decomposition ===")
print(
    pairs.groupby("counterfactual_type")
    .agg(
        proteins=("protein_22", "nunique"),
        positive_pairs=("go_id_2022", "size")
    )
)

print("\n=== Namespace by source ===")
print(
    pd.crosstab(
        pairs["namespace"],
        pairs["counterfactual_type"]
    )
)

print("\n=== Historical-novel and vocab coverage ===")
eligible = pairs[
    pairs["in_model_vocab"].eq(True) &
    pairs["not_implied_by_2022_ipr2go"].eq(True) &
    pairs["not_in_2022_self_go_closure"].eq(True)
]

print(
    eligible.groupby("namespace")
    .agg(
        proteins=("protein_22", "nunique"),
        positive_pairs=("go_id_2022", "size"),
        unique_go=("go_id_2022", "nunique")
    )
)

In [48]:
import json
from pathlib import Path

root = Path("data/dark_weak_test/unified")

with open(root / "summary.json") as f:
    summary = json.load(f)

keys = [
    "n_future_ipr2go_delta_proteins",
    "n_future_ipr2go_crosssource_novel_proteins",
    "n_model_evaluable_experimental_swissprot_novel_proteins",
    "n_model_evaluable_ipr2go_crosssource_novel_proteins",
    "n_model_evaluable_union_novel_proteins",
    "n_model_evaluable_new_ipr2go_proteins_beyond_experimental",
    "n_future_overlap_direct_pairs",
]

for key in keys:
    print(f"{key}: {summary[key]}")
    

In [49]:
import json
from pathlib import Path

root = Path("./dark_weak_test/unified")

with open(root / "summary.json") as f:
    summary = json.load(f)

keys = [
    "n_future_ipr2go_delta_proteins",
    "n_future_ipr2go_crosssource_novel_proteins",
    "n_model_evaluable_experimental_swissprot_novel_proteins",
    "n_model_evaluable_ipr2go_crosssource_novel_proteins",
    "n_model_evaluable_union_novel_proteins",
    "n_model_evaluable_new_ipr2go_proteins_beyond_experimental",
    "n_future_overlap_direct_pairs",
]

for key in keys:
    print(f"{key}: {summary[key]}")
    

In [50]:
import pandas as pd

stats = pd.read_csv(
    "./dark_weak_test/unified/tables/matrix_summary.tsv",
    sep="\t"
)

print(stats.to_string(index=False))

In [51]:
import pandas as pd

stats = pd.read_csv(
    "./dark_weak_test/unified/tables/ipr2go_transition_summary.tsv",
    sep="\t"
)

print(stats.to_string(index=False))

In [52]:
import pickle

with open("./unidata_with_dark_weak_eval.pkl", "rb") as f:
    metadata = pickle.load(f)

for split in [
    "dark_weak_test",
    "dark_weak_test_exp",
    "dark_weak_test_ipr2go",
    "dark_weak_test_exp_direct",
]:
    data = metadata[split]["biological_process"]

    direct = data["annotations"]
    propagated = data["prop_annotations"]

    assert len(data["proteins"]) == len(direct) == len(propagated)
    assert all(
        set(d).issubset(set(p))
        for d, p in zip(direct, propagated)
    )

    print(
        split,
        "proteins:", len(data["proteins"]),
        "direct:", sum(map(len, direct)),
        "propagated:", sum(map(len, propagated))
    )
    

In [53]:
metadata.keys()

In [54]:
with open('./unidata_with_dark_weak_eval.pkl','rb') as f:
    meta=pickle.load(f)
for split in ['dark_weak_test_direct', 'dark_weak_test_exp_direct',
              'dark_weak_test_ipr2go_direct', 'dark_weak_test_2022_ipr2go_direct']:
    print(split)
    for ns in ('biological_process','molecular_function','cellular_component'):
        p=meta[split][ns]['proteins']
        direct=meta[split][ns]['annotations']
        prop=meta[split][ns]['prop_annotations']
        assert len(p)==len(direct)==len(prop)
        assert all(set(a).issubset(b) for a,b in zip(direct,prop))
        print(' ',ns,len(p),'direct pairs',sum(map(len,direct)), 'ancestor pairs',sum(map(len,prop)))
        

In [55]:
%notebook notebooks/build_dark_weak_test_set.ipynb